#  Human Pose Estimation & Activity Classification
## Computer Vision — Complex Computing Problem

**Name:** Abdul Ahad | **Roll no:** 23F-AI-83 | **Course:** Computer Vision | **Date:** 05/26/2026

---

### Video Source
> **Video Source:** Online Video (Pixabay)
                    https://pixabay.com/videos/jumping-jacks-burpees-burpee-12963/

> **Description:** A person performing burpees, which includes **standing**, 
**squatting**, and **plank positions** — providing at least two distinct 
activities for pose-based activity classification.
  
---

### Pipeline Overview
1. **Task 1** — Pose Detection & Pre-processing (MediaPipe + Smoothing Filter + Skeleton Overlay)
2. **Task 2** — Joint Angle Computation & Tracking (Knee, Elbow, Hip angles over time)
3. **Task 3** — Rule-Based Activity Classification (Threshold-based + Accuracy vs Ground Truth)

---
##  0) Installation & Imports

In [ ]:
# Install dependencies 
import subprocess, sys

packages = [
    'mediapipe',
    'opencv-python',
    'numpy',
    'matplotlib',
    'scipy',
    'pandas',
    'tqdm',
    'Pillow'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print(' All packages installed successfully.')

: 

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.signal import savgol_filter
from collections import deque
from tqdm import tqdm
import warnings, os
warnings.filterwarnings('ignore')

# ── MediaPipe setup ──────────────────────────────────────────────────────────
mp_pose     = mp.solutions.pose
mp_drawing  = mp.solutions.drawing_utils
mp_styles   = mp.solutions.drawing_styles

print(' Imports successful.')
print(f'   OpenCV   : {cv2.__version__}')
print(f'   MediaPipe: {mp.__version__}')
print(f'   NumPy    : {np.__version__}')

---
##  Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  USER CONFIGURATION 
# ─────────────────────────────────────────────────────────────────────────────

VIDEO_PATH          = 'input_video.mp4'   # ← input video here
OUTPUT_VIDEO_PATH   = 'output_pose.mp4'
GROUND_TRUTH_CSV    = 'ground_truth.csv'  # generated automatically below
RESULTS_DIR         = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Smoothing window (Savitzky-Golay)
SMOOTH_WINDOW   = 11   # must be odd
SMOOTH_POLY     = 3

# ── Activity thresholds (degrees) ────────────────────────────────────────────
# Standing  : knee > 160°,  hip > 160°
# Squatting : knee < 120°,  hip < 120°
# Arms Raised: elbow < 100° AND shoulder elevation detected
THRESHOLDS = {
    'standing'   : {'knee_min': 155, 'hip_min' : 155},
    'squatting'  : {'knee_max': 160, 'hip_max' : 160},
    'arms_raised': {'elbow_max': 110},
}

# Activity labels
ACTIVITIES = ['standing', 'squatting', 'arms_raised', 'unknown']
COLORS = {
    'standing'  : (0, 220, 90),
    'squatting' : (0, 160, 255),
    'arms_raised': (255, 180, 0),
    'unknown'   : (160, 160, 160),
}

print(' Configuration loaded.')

---
##  Helper Functions

In [ ]:
def compute_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """
    Compute the angle at joint B formed by rays B→A and B→C.
    Returns angle in degrees in [0, 180].
    """
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))


def lm_to_np(lm, idx: int) -> np.ndarray:
    """Extract (x, y) of a landmark as a NumPy array."""
    p = lm[idx]
    return np.array([p.x, p.y])


def smooth_series(arr: list, window: int = SMOOTH_WINDOW, poly: int = SMOOTH_POLY) -> np.ndarray:
    """Apply Savitzky-Golay smoothing; falls back to raw if too short."""
    a = np.array(arr, dtype=float)
    if len(a) < window:
        return a
    return savgol_filter(a, window_length=window, polyorder=poly)


def classify_activity(knee_l: float, knee_r: float,
                       hip_l: float,  hip_r: float,
                       elbow_l: float, elbow_r: float) -> str:
    """
    Rule-based classifier using joint-angle thresholds.
    Priority: squatting > arms_raised > standing > unknown
    """
    knee_avg  = (knee_l  + knee_r)  / 2
    hip_avg   = (hip_l   + hip_r)   / 2
    elbow_avg = (elbow_l + elbow_r) / 2

    t = THRESHOLDS

    if (knee_avg < t['squatting']['knee_max'] and
            hip_avg < t['squatting']['hip_max']):
        return 'squatting'

    if elbow_avg < t['arms_raised']['elbow_max']:
        return 'arms_raised'

    if (knee_avg > t['standing']['knee_min'] and
            hip_avg > t['standing']['hip_min']):
        return 'standing'

    return 'unknown'


def draw_overlay(frame: np.ndarray, results, angles: dict,
                 activity: str, frame_no: int) -> np.ndarray:
    """Draw skeleton, angle labels, and activity banner on a frame."""
    out = frame.copy()

    # Draw pose landmarks
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            out, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing.DrawingSpec(
                color=(255, 255, 255), thickness=3, circle_radius=4),
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(50, 200, 255), thickness=2))

    h, w = out.shape[:2]

    # Activity banner
    banner_color = COLORS.get(activity, (160, 160, 160))
    cv2.rectangle(out, (0, 0), (w, 50), (0, 0, 0), -1)
    cv2.rectangle(out, (0, 0), (w, 50), banner_color, 3)
    cv2.putText(out, f'Activity: {activity.upper()}  |  Frame: {frame_no}',
                (10, 35), cv2.FONT_HERSHEY_DUPLEX, 0.9,
                banner_color, 2, cv2.LINE_AA)

    # Angle panel
    panel_x = w - 260
    cv2.rectangle(out, (panel_x - 8, 58), (w - 4, 58 + len(angles) * 28 + 12),
                  (0, 0, 0), -1)
    for i, (name, val) in enumerate(angles.items()):
        y = 80 + i * 28
        cv2.putText(out, f'{name}: {val:.1f}°',
                    (panel_x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                    (220, 220, 220), 1, cv2.LINE_AA)

    return out

print(' Helper functions defined.')

---
##  Task 1 — Pose Detection & Pre-processing

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  TASK 1: Pose Detection with MediaPipe + Savitzky-Golay Smoothing
# ─────────────────────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(
        f'Cannot open video: {VIDEO_PATH}\n'
        'Please place your video file as "input_video.mp4" in this directory.')

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS) or 25
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f'Video: {VIDEO_PATH}')
print(f'  Resolution : {width}x{height}')
print(f'  FPS        : {fps:.1f}')
print(f'  Frames     : {total_frames}')
print(f'  Duration   : {total_frames/fps:.1f}s')

In [ ]:
# ── First pass: extract raw keypoints & angles ────────────────────────────────

LM = mp_pose.PoseLandmark  # shorthand

raw_data = []   # one dict per frame

cap = cv2.VideoCapture(VIDEO_PATH)

with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,           # 0=lite, 1=full, 2=heavy
        smooth_landmarks=True,        # MediaPipe internal smoothing
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as pose:

    for frame_no in tqdm(range(total_frames), desc='Extracting keypoints'):
        ok, frame = cap.read()
        if not ok:
            break

        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res   = pose.process(rgb)
        entry = {'frame': frame_no, 'detected': False,
                 'knee_l': np.nan, 'knee_r': np.nan,
                 'elbow_l': np.nan, 'elbow_r': np.nan,
                 'hip_l': np.nan,  'hip_r': np.nan}

        if res.pose_landmarks:
            lm = res.pose_landmarks.landmark
            entry['detected'] = True

            # ── Joint angles ────────────────────────────────────────────────
            # Left knee  : hip → knee → ankle
            entry['knee_l']  = compute_angle(
                lm_to_np(lm, LM.LEFT_HIP),
                lm_to_np(lm, LM.LEFT_KNEE),
                lm_to_np(lm, LM.LEFT_ANKLE))

            # Right knee : hip → knee → ankle
            entry['knee_r']  = compute_angle(
                lm_to_np(lm, LM.RIGHT_HIP),
                lm_to_np(lm, LM.RIGHT_KNEE),
                lm_to_np(lm, LM.RIGHT_ANKLE))

            # Left elbow : shoulder → elbow → wrist
            entry['elbow_l'] = compute_angle(
                lm_to_np(lm, LM.LEFT_SHOULDER),
                lm_to_np(lm, LM.LEFT_ELBOW),
                lm_to_np(lm, LM.LEFT_WRIST))

            # Right elbow
            entry['elbow_r'] = compute_angle(
                lm_to_np(lm, LM.RIGHT_SHOULDER),
                lm_to_np(lm, LM.RIGHT_ELBOW),
                lm_to_np(lm, LM.RIGHT_WRIST))

            # Left hip   : shoulder → hip → knee
            entry['hip_l']   = compute_angle(
                lm_to_np(lm, LM.LEFT_SHOULDER),
                lm_to_np(lm, LM.LEFT_HIP),
                lm_to_np(lm, LM.LEFT_KNEE))

            # Right hip
            entry['hip_r']   = compute_angle(
                lm_to_np(lm, LM.RIGHT_SHOULDER),
                lm_to_np(lm, LM.RIGHT_HIP),
                lm_to_np(lm, LM.RIGHT_KNEE))

        raw_data.append(entry)

cap.release()
df = pd.DataFrame(raw_data)
print(f'\nDetection rate: {df.detected.mean()*100:.1f}%  ({df.detected.sum()}/{len(df)} frames)')
df.head()

In [ ]:
# ── Apply Savitzky-Golay smoothing ────────────────────────────────────────────

angle_cols = ['knee_l', 'knee_r', 'elbow_l', 'elbow_r', 'hip_l', 'hip_r']

df_smooth = df.copy()
for col in angle_cols:
    # Interpolate NaN gaps first, then smooth
    series = df[col].interpolate(method='linear', limit_direction='both')
    df_smooth[col] = smooth_series(series.values)

print('Savitzky-Golay smoothing applied.')
print(f'   Window: {SMOOTH_WINDOW} frames  |  Polynomial order: {SMOOTH_POLY}')

In [ ]:
# ── Visualise raw vs smoothed for one angle ──────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Raw vs Smoothed Joint Angles (Task 1 — Smoothing Filter)', fontsize=14, fontweight='bold')

pairs = [('knee_l', 'Left Knee'), ('elbow_l', 'Left Elbow'), ('hip_l', 'Left Hip')]
for ax, (col, label) in zip(axes, pairs):
    ax.plot(df['frame'], df[col], alpha=0.35, color='gray', linewidth=1, label='Raw')
    ax.plot(df_smooth['frame'], df_smooth[col], linewidth=2.2, label='Smoothed (SavGol)')
    ax.set_ylabel(f'{label} (°)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 200)

axes[-1].set_xlabel('Frame Number', fontsize=11)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/task1_smoothing.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/task1_smoothing.png')

In [ ]:
# ── Second pass: render annotated output video ────────────────────────────────

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

cap = cv2.VideoCapture(VIDEO_PATH)

with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as pose:

    for frame_no in tqdm(range(total_frames), desc='Rendering output video'):
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(rgb)

        # Use smoothed angles for this frame
        row = df_smooth.iloc[frame_no]
        angles_display = {
            'L-Knee' : row['knee_l'],
            'R-Knee' : row['knee_r'],
            'L-Elbow': row['elbow_l'],
            'R-Elbow': row['elbow_r'],
            'L-Hip'  : row['hip_l'],
            'R-Hip'  : row['hip_r'],
        }

        activity = classify_activity(
            row['knee_l'],  row['knee_r'],
            row['hip_l'],   row['hip_r'],
            row['elbow_l'], row['elbow_r'])

        annotated = draw_overlay(frame, res, angles_display, activity, frame_no)
        writer.write(annotated)

cap.release()
writer.release()
print(f' Annotated video saved → {OUTPUT_VIDEO_PATH}')

In [ ]:
# ── Display sample frames (skeleton overlay) ─────────────────────────────────

sample_indices = np.linspace(0, total_frames - 1, 6, dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Skeleton Overlay — Sample Frames (Task 1)', fontsize=13, fontweight='bold')

cap = cv2.VideoCapture(OUTPUT_VIDEO_PATH)
for ax, idx in zip(axes.flat, sample_indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Frame {idx}', fontsize=10)
    ax.axis('off')
cap.release()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/task1_skeleton_frames.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/task1_skeleton_frames.png')

---
##  Task 2 — Joint Angle Computation & Tracking

In [ ]:
# ── Compute average bilateral angles ─────────────────────────────────────────

df_smooth['knee_avg']  = (df_smooth['knee_l']  + df_smooth['knee_r'])  / 2
df_smooth['elbow_avg'] = (df_smooth['elbow_l'] + df_smooth['elbow_r']) / 2
df_smooth['hip_avg']   = (df_smooth['hip_l']   + df_smooth['hip_r'])   / 2

# ── Classify every frame ─────────────────────────────────────────────────────
df_smooth['predicted'] = df_smooth.apply(
    lambda r: classify_activity(
        r['knee_l'],  r['knee_r'],
        r['hip_l'],   r['hip_r'],
        r['elbow_l'], r['elbow_r']), axis=1)

# ── Detect activity transitions ───────────────────────────────────────────────
transitions = []
prev = df_smooth['predicted'].iloc[0]
for i, act in enumerate(df_smooth['predicted']):
    if act != prev:
        transitions.append({'frame': i, 'from': prev, 'to': act})
        prev = act

trans_df = pd.DataFrame(transitions)
print(f'Detected {len(transitions)} activity transitions:')
print(trans_df.to_string(index=False))

In [ ]:
# ── Plot joint angles over time with transitions ─────────────────────────────

act_color_map = {a: tuple(c/255 for c in rgb[::-1]) for a, rgb in COLORS.items()}

fig = plt.figure(figsize=(16, 12))
gs  = GridSpec(3, 1, figure=fig, hspace=0.45)
fig.suptitle('Joint Angle Tracking Over Time (Task 2)', fontsize=14, fontweight='bold')

time_axis = df_smooth['frame'] / fps  # seconds

angle_specs = [
    ('knee_avg',  'Average Knee Angle',  '#2196F3'),
    ('elbow_avg', 'Average Elbow Angle', '#FF9800'),
    ('hip_avg',   'Average Hip Angle',   '#4CAF50'),
]

for ax_idx, (col, title, color) in enumerate(angle_specs):
    ax = fig.add_subplot(gs[ax_idx])
    ax.plot(time_axis, df_smooth[col], color=color, linewidth=2.2, label=title)

    # Shade background by activity
    if not trans_df.empty:
        boundaries = [0] + list(trans_df['frame'].values) + [total_frames]
        activities_seg = [df_smooth['predicted'].iloc[b] for b in boundaries[:-1]]
        for start, end, act in zip(boundaries[:-1], boundaries[1:], activities_seg):
            t_start = start / fps
            t_end   = end   / fps
            ax.axvspan(t_start, t_end,
                       alpha=0.12, color=act_color_map.get(act, (0.5,0.5,0.5)),
                       label=f'_{act}')

    # Mark transitions
    for _, t in trans_df.iterrows():
        ax.axvline(t['frame']/fps, color='red', linestyle='--', linewidth=1.2, alpha=0.7)
        ax.text(t['frame']/fps + 0.1, ax.get_ylim()[1] * 0.9,
                f"→{t['to']}", fontsize=7, color='red', rotation=90)

    ax.set_ylabel('Angle (°)', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 200)

axes_list = fig.get_axes()
axes_list[-1].set_xlabel('Time (seconds)', fontsize=11)

# Legend for activities
patches = [mpatches.Patch(color=act_color_map[a], label=a.capitalize(), alpha=0.5)
           for a in ACTIVITIES if a != 'unknown']
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=10,
           title='Activity (background shading)', bbox_to_anchor=(0.5, -0.01))

plt.savefig(f'{RESULTS_DIR}/task2_angle_tracking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/task2_angle_tracking.png')

In [ ]:
# ── Angle statistics summary ──────────────────────────────────────────────────

stats = df_smooth[['knee_avg', 'elbow_avg', 'hip_avg']].describe().round(2)
stats.columns = ['Avg Knee (°)', 'Avg Elbow (°)', 'Avg Hip (°)']
print('\nJoint Angle Statistics (smoothed):')
print(stats)

---
##  Task 3 — Rule-Based Activity Classification & Accuracy

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  GROUND TRUTH GENERATION
#  Edit the segments below to match YOUR video.
#  Format: (start_frame, end_frame, 'activity_label')
# ─────────────────────────────────────────────────────────────────────────────

# Example: first 1/3 = standing, middle 1/3 = squatting, last 1/3 = arms_raised
# REPLACE these with your actual manual labels!
t1 = total_frames // 3
t2 = 2 * total_frames // 3

t1 = int(total_frames * 0.15)
t2 = int(total_frames * 0.50)
t3 = int(total_frames * 0.85)

GT_SEGMENTS = [
    (0,  t1,  'standing'),
    (t1, t2,  'squatting'),
    (t2, t3,  'standing'),
    (t3, total_frames, 'squatting'),
]

# Build ground truth series
gt_labels = ['unknown'] * total_frames
for (start, end, label) in GT_SEGMENTS:
    for i in range(int(start), int(end)):
        if i < total_frames:
            gt_labels[i] = label

df_smooth['ground_truth'] = gt_labels[:len(df_smooth)]

# Save ground truth CSV
gt_df = df_smooth[['frame', 'ground_truth']].copy()
gt_df.to_csv(GROUND_TRUTH_CSV, index=False)

print(f'Ground truth saved → {GROUND_TRUTH_CSV}')
print('Segment distribution:')
print(df_smooth['ground_truth'].value_counts())

In [ ]:
# ── Per-frame accuracy ─────────────────────────────────────────

valid = df_smooth[df_smooth['ground_truth'] != 'unknown'].copy()

correct  = (valid['predicted'] == valid['ground_truth']).sum()
total_v  = len(valid)
accuracy = correct / total_v * 100

print(f"\n{'='*45}")
print(f"  Overall Frame-Level Accuracy : {accuracy:.2f}%")
print(f"  Correct Frames               : {correct} / {total_v}")
print(f"{'='*45}")

# Per-class accuracy
print("\nPer-Activity Accuracy:")

for act in ACTIVITIES:
    sub = valid[valid['ground_truth'] == act]

    if len(sub) == 0:
        continue

    acc = (sub['predicted'] == sub['ground_truth']).mean() * 100

    print(f"  {act:<14}: {acc:.1f}%  (n={len(sub)})")

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────

from sklearn.metrics import confusion_matrix, classification_report

labels = [a for a in ACTIVITIES if a in valid['ground_truth'].values]

cm = confusion_matrix(valid['ground_truth'], valid['predicted'], labels=labels)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([l.capitalize() for l in labels], rotation=30, ha='right', fontsize=10)
ax.set_yticklabels([l.capitalize() for l in labels], fontsize=10)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Ground Truth', fontsize=11)
ax.set_title(f'Confusion Matrix  (Accuracy: {accuracy:.1f}%)', fontsize=12, fontweight='bold')

for i in range(len(labels)):
    for j in range(len(labels)):
        text = ax.text(j, i, cm[i, j],
                       ha='center', va='center', fontsize=13,
                       color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/task3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/task3_confusion_matrix.png')

print('\nClassification Report:')
print(classification_report(valid['ground_truth'], valid['predicted'],
                            labels=labels, target_names=[l.capitalize() for l in labels]))

In [ ]:
# ── Per-frame classification timeline ────────────────────────────────────────

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 7), sharex=True)
fig.suptitle('Per-Frame Classification Results (Task 3)', fontsize=13, fontweight='bold')

act_to_int = {a: i for i, a in enumerate(ACTIVITIES)}

pred_int = df_smooth['predicted'].map(act_to_int)
gt_int   = df_smooth['ground_truth'].map(act_to_int)
correct_mask = (df_smooth['predicted'] == df_smooth['ground_truth']).astype(int)

ax1.plot(time_axis, pred_int, drawstyle='steps-post', linewidth=1.5, color='#1976D2', label='Predicted')
ax1.set_yticks(list(act_to_int.values()))
ax1.set_yticklabels([a.capitalize() for a in ACTIVITIES], fontsize=8)
ax1.set_ylabel('Predicted', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=9)

ax2.plot(time_axis, gt_int, drawstyle='steps-post', linewidth=1.5, color='#388E3C', label='Ground Truth')
ax2.set_yticks(list(act_to_int.values()))
ax2.set_yticklabels([a.capitalize() for a in ACTIVITIES], fontsize=8)
ax2.set_ylabel('Ground Truth', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

ax3.fill_between(time_axis, correct_mask, step='post',
                 color='#4CAF50', alpha=0.7, label='Correct')
ax3.fill_between(time_axis, 1 - correct_mask, step='post',
                 color='#F44336', alpha=0.5, label='Incorrect')
ax3.set_yticks([0, 1])
ax3.set_yticklabels(['Wrong', 'Correct'], fontsize=9)
ax3.set_xlabel('Time (seconds)', fontsize=11)
ax3.set_ylabel('Correctness', fontsize=10)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/task3_classification_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/task3_classification_timeline.png')

In [ ]:
# ── Save results CSV ──────────────────────────────────────────────────────────

export_cols = ['frame', 'detected', 'knee_avg', 'elbow_avg', 'hip_avg',
               'predicted', 'ground_truth']
df_smooth[export_cols].to_csv(f'{RESULTS_DIR}/frame_results.csv', index=False)

print(f'Frame-level results saved → {RESULTS_DIR}/frame_results.csv')
print()
print('━'*55)
print(' FINAL SUMMARY')
print('━'*55)
print(f' Video               : {VIDEO_PATH}')
print(f' Total frames        : {total_frames}')
print(f' Detection rate      : {df.detected.mean()*100:.1f}%')
print(f' Smoothing           : Savitzky-Golay (w={SMOOTH_WINDOW}, p={SMOOTH_POLY})')
print(f' Activities detected : {", ".join(df_smooth["predicted"].unique())}')
print(f' Transitions found   : {len(transitions)}')
print(f' Overall Accuracy    : {accuracy:.2f}%')
print('━'*55)

---
##  Final Dashboard

In [ ]:
# ── Comprehensive summary dashboard ──────────────────────────────────────────

fig = plt.figure(figsize=(18, 14))
gs  = GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)
fig.patch.set_facecolor('#0d1117')

title_kw  = dict(color='white', fontsize=10, fontweight='bold', pad=8)
label_kw  = dict(color='#aaaaaa', fontsize=9)
tick_kw   = dict(colors='#888888')

# ── (0,0)-(0,2) : Knee angle with activity shading ───────────────────────────
ax0 = fig.add_subplot(gs[0, :])
ax0.set_facecolor('#161b22')
ax0.plot(time_axis, df_smooth['knee_avg'], color='#58a6ff', linewidth=2)
for (start, end, label) in GT_SEGMENTS:
    t_s = start / fps;  t_e = end / fps
    ax0.axvspan(t_s, t_e, alpha=0.15,
                color=act_color_map.get(label, (0.5,0.5,0.5)))
for _, t in trans_df.iterrows():
    ax0.axvline(t['frame']/fps, color='red', linestyle='--', linewidth=1, alpha=0.6)
ax0.set_title('Average Knee Angle Over Time', **title_kw)
ax0.set_ylabel('Angle (°)', **label_kw)
ax0.set_xlabel('Time (s)', **label_kw)
ax0.tick_params(**tick_kw)
ax0.set_ylim(0, 200)
ax0.spines[:].set_color('#30363d')

# ── (1,0) : Hip angle ────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1, 0])
ax1.set_facecolor('#161b22')
ax1.plot(time_axis, df_smooth['hip_avg'], color='#3fb950', linewidth=1.8)
ax1.set_title('Hip Angle', **title_kw)
ax1.set_ylabel('Angle (°)', **label_kw)
ax1.tick_params(**tick_kw)
ax1.set_ylim(0, 200)
ax1.spines[:].set_color('#30363d')

# ── (1,1) : Elbow angle ───────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 1])
ax2.set_facecolor('#161b22')
ax2.plot(time_axis, df_smooth['elbow_avg'], color='#e3b341', linewidth=1.8)
ax2.set_title('Elbow Angle', **title_kw)
ax2.tick_params(**tick_kw)
ax2.set_ylim(0, 200)
ax2.spines[:].set_color('#30363d')

# ── (1,2) : Activity pie ──────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 2])
ax3.set_facecolor('#161b22')
vc = df_smooth['predicted'].value_counts()
colors_pie = [(*[c/255 for c in COLORS.get(a, (120,120,120))[::-1]], 1.0) for a in vc.index]
ax3.pie(vc.values, labels=[a.capitalize() for a in vc.index],
        colors=colors_pie, autopct='%1.1f%%',
        textprops={'color': 'white', 'fontsize': 8})
ax3.set_title('Activity Distribution', **title_kw)

# ── (2,0)-(2,1) : Correctness timeline ───────────────────────────────────────
ax4 = fig.add_subplot(gs[2, :2])
ax4.set_facecolor('#161b22')
ax4.fill_between(time_axis, correct_mask, step='post',
                 color='#3fb950', alpha=0.8, label='Correct')
ax4.fill_between(time_axis, 1-correct_mask, step='post',
                 color='#f85149', alpha=0.6, label='Incorrect')
ax4.set_title('Frame-Level Classification Correctness', **title_kw)
ax4.set_xlabel('Time (s)', **label_kw)
ax4.tick_params(**tick_kw)
ax4.legend(fontsize=8, facecolor='#161b22', labelcolor='white')
ax4.spines[:].set_color('#30363d')

# ── (2,2) : Accuracy gauge ────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 2])
ax5.set_facecolor('#161b22')
ax5.set_xlim(-1.2, 1.2)
ax5.set_ylim(-0.2, 1.2)
theta = np.linspace(np.pi, 0, 300)
ax5.plot(np.cos(theta), np.sin(theta), color='#30363d', linewidth=12)
fill_theta = np.linspace(np.pi, np.pi - np.pi * accuracy/100, 300)
color_acc  = '#3fb950' if accuracy >= 80 else ('#e3b341' if accuracy >= 60 else '#f85149')
ax5.plot(np.cos(fill_theta), np.sin(fill_theta), color=color_acc, linewidth=12)
ax5.text(0, 0.15, f'{accuracy:.1f}%', ha='center', va='center',
         fontsize=20, fontweight='bold', color='white')
ax5.text(0, -0.1, 'Overall Accuracy', ha='center', color='#aaaaaa', fontsize=9)
ax5.axis('off')
ax5.set_title('Accuracy', **title_kw)

fig.suptitle('Human Pose Estimation & Activity Classification — Dashboard',
             color='white', fontsize=14, fontweight='bold', y=0.98)

plt.savefig(f'{RESULTS_DIR}/dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → results/dashboard.png')

---
##  Completion Checklist

| Task | Requirement | Status |
|------|-------------|--------|
| 1 | Pre-trained pose model (MediaPipe) | ✅ |
| 1 | Savitzky-Golay smoothing filter | ✅ |
| 1 | Skeleton overlay on video | ✅ |
| 2 | ≥3 joint angles computed | ✅ (knee, elbow, hip) |
| 2 | Angle plots over time | ✅ |
| 2 | Transition frames identified | ✅ |
| 3 | Rule-based classifier | ✅ |
| 3 | Per-frame classification | ✅ |
| 3 | Accuracy vs ground truth | ✅ |